<a href="https://colab.research.google.com/github/mrcheong200/RNN/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN 迴圈神經網路

## 0.匯入套件

In [ ]:
# 匯入 PyTorch 主套件
# torch 用來處理 tensor、數值運算、模型訓練等
import torch

# 匯入神經網路模組，通常會簡寫成 nn
# 裡面有 RNN、Linear、Embedding、Loss function 等常用元件
import torch.nn as nn

# 從 torch.utils.data 匯入 Dataset 和 DataLoader
# Dataset: 定義資料集格式
# DataLoader: 幫你把資料切成 batch、打亂順序、逐批讀取
from torch.utils.data import Dataset, DataLoader

## 1. 準備資料

In [ ]:
# 定義訓練文字資料
# 這裡用三引號字串，方便直接寫多行文字
# lower() 會把所有英文字母轉成小寫，減少 vocab 複雜度
text = """
hello world.
this is a simple example.
rnn can learn next character prediction.
hello world.
this is a simple example.
rnn can learn next character prediction.
""".lower()

# set(text) 會取出字串中所有「不重複」的字元
# list(...) 把 set 轉成串列
# sorted(...) 把字元排序，讓 vocab 順序固定
chars = sorted(list(set(text)))

# vocab_size 表示總共有多少種不同字元
# 例如可能包含：
# 空白、換行、h、e、l、o、w、r、d、.、a...
vocab_size = len(chars)

# 建立「字元 -> 整數索引」的字典
# enumerate(chars) 會產生 (index, char)
# 例如 {'\n': 0, ' ': 1, '.': 2, 'a': 3, ...}
char_to_idx = {ch: i for i, ch in enumerate(chars)}

# 建立「整數索引 -> 字元」的反向字典
# 之後模型輸出的是 index，要靠這個字典轉回字元
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# 印出所有字元表，方便你確認資料內容
print("Vocabulary:", chars)

# 印出 vocab 大小
print("Vocab size:", vocab_size)

# 把整段文字 text 轉成數字序列
# 對 text 中每個字元 ch，查 char_to_idx[ch]
# 例如 "hello" 可能被轉成 [6, 4, 8, 8, 10]
encoded_text = [char_to_idx[ch] for ch in text]

Vocabulary: ['\n', ' ', '.', 'a', 'c', 'd', 'e', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'w', 'x']
Vocab size: 19


## 2. 建立 Dataset

In [ ]:
# 自訂一個字元資料集類別，繼承 PyTorch 的 Dataset
class CharDataset(Dataset):
    # 初始化函式
    # data: 整段已編碼的文字（整數序列）
    # seq_len: 每次取出的輸入序列長度
    def __init__(self, data, seq_len):
        # 把傳進來的 data 存成物件屬性
        self.data = data

        # 把傳進來的序列長度存成物件屬性
        self.seq_len = seq_len

    # 定義資料集總長度
    # 這會告訴 DataLoader 總共有多少筆樣本
    def __len__(self):
        # 如果總資料長度是 N，每個樣本需要 seq_len + 1 個位置
        # 因為 x 用 seq_len 個字元，y 也要往後對齊 seq_len 個字元
        return len(self.data) - self.seq_len

    # 定義如何取得第 idx 筆資料
    def __getitem__(self, idx):
        # 取出輸入序列 x
        # 從 idx 開始，取 seq_len 個字元
        x = self.data[idx: idx + self.seq_len]

        # 取出目標序列 y
        # 往右偏移一格，所以從 idx+1 開始
        # 一樣取 seq_len 個字元
        y = self.data[idx + 1: idx + self.seq_len + 1]

        # 把 x 轉成 LongTensor
        # dtype=torch.long 是因為 embedding 和分類標籤都需要整數 index
        x_tensor = torch.tensor(x, dtype=torch.long)

        # 把 y 也轉成 LongTensor
        y_tensor = torch.tensor(y, dtype=torch.long)

        # 回傳一筆訓練資料：(輸入序列, 目標序列)
        return x_tensor, y_tensor


# 設定序列長度
# 表示每次拿 20 個字元當作模型輸入
seq_len = 20

# 建立資料集物件
dataset = CharDataset(encoded_text, seq_len)

# 建立 DataLoader
# batch_size=16 表示每次訓練抓 16 筆資料
# shuffle=True 表示每個 epoch 都打亂資料順序，有助訓練
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

## 3. 定義 RNN 模型


In [ ]:
# 定義一個字元級 RNN 模型
# 繼承 nn.Module，這是 PyTorch 所有模型的基底類別
class CharRNN(nn.Module):
    # 初始化模型
    # vocab_size: 總字元數
    # embed_size: embedding 向量維度
    # hidden_size: RNN 隱藏狀態維度
    def __init__(self, vocab_size, embed_size, hidden_size):
        # 呼叫父類別 nn.Module 的初始化
        super().__init__()

        # 建立 embedding 層
        # 功能：把每個字元的 index 映射成一個稠密向量
        # 輸入 shape: (batch, seq_len)
        # 輸出 shape: (batch, seq_len, embed_size)
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # 建立 RNN 層
        # input_size=embed_size，因為 RNN 接收的是 embedding 向量
        # hidden_size=hidden_size，代表隱藏狀態維度
        # batch_first=True 代表輸入格式用 (batch, seq_len, feature)
        self.rnn = nn.RNN(
            input_size=embed_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # 建立全連接層（線性層）
        # 把每個時間步的 hidden state 映射成 vocab_size 維
        # 也就是對每個字元類別產生一個分數（logits）
        self.fc = nn.Linear(hidden_size, vocab_size)

    # 定義前向傳播
    # x: 輸入字元 index，shape = (batch, seq_len)
    # h: 初始 hidden state，預設為 None
    def forward(self, x, h=None):
        # 先把 index 轉成 embedding 向量
        # x shape 從 (batch, seq_len)
        # 變成 (batch, seq_len, embed_size)
        x = self.embedding(x)

        # 把 embedding 序列送進 RNN
        # out: 每個時間步的 hidden output
        # h: 最後一個時間步的 hidden state
        # out shape = (batch, seq_len, hidden_size)
        # h shape = (1, batch, hidden_size)
        out, h = self.rnn(x, h)

        # 把每個時間步的 hidden state 丟進全連接層
        # out shape 變成 (batch, seq_len, vocab_size)
        out = self.fc(out)

        # 回傳每個時間步的分類分數，以及最後 hidden state
        return out, h


# 判斷是否有 GPU 可用
# 如果有 CUDA，就用 GPU 訓練
# 沒有就退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 建立模型實例
# vocab_size: 字元種類數
# embed_size=16: 每個字元先轉成 16 維向量
# hidden_size=64: RNN 的隱藏狀態用 64 維
model = CharRNN(
    vocab_size=vocab_size,
    embed_size=16,
    hidden_size=64
).to(device)  # 把模型移到 device 上（CPU 或 GPU）

# 定義損失函數
# CrossEntropyLoss 適合做多分類問題
# 這裡每個時間步都要從 vocab_size 種字元中選出正確答案
criterion = nn.CrossEntropyLoss()

# 定義優化器 Adam
# model.parameters() 代表要更新模型中的所有參數
# lr=0.01 是學習率
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 4. 訓練模型


In [ ]:
# 設定總共訓練幾個 epoch
# 一個 epoch 表示整份資料完整跑過一次
epochs = 50

# 外層迴圈：每次跑一個 epoch
for epoch in range(epochs):
    # 切換模型到訓練模式
    # 某些層如 Dropout、BatchNorm 在 train/eval 模式行為不同
    model.train()

    # 累積本 epoch 的總 loss
    total_loss = 0

    # 從 dataloader 一批一批取資料
    # x_batch shape: (batch_size, seq_len)
    # y_batch shape: (batch_size, seq_len)
    for x_batch, y_batch in dataloader:
        # 把輸入資料移到 device 上
        x_batch = x_batch.to(device)

        # 把目標資料移到 device 上
        y_batch = y_batch.to(device)

        # 把前一次 batch 累積的梯度清空
        # PyTorch 預設梯度會累加，所以每次 backward 前要歸零
        optimizer.zero_grad()

        # 前向傳播
        # outputs shape = (batch, seq_len, vocab_size)
        # _ 代表我們這裡先不使用最後 hidden state
        outputs, _ = model(x_batch)

        # 計算 loss
        # CrossEntropyLoss 期望輸入 shape 是 (N, C)
        # 其中 N 是樣本數，C 是類別數
        # 所以要把 outputs 從 (batch, seq_len, vocab_size)
        # 攤平成 (batch * seq_len, vocab_size)
        #
        # y_batch 原本是 (batch, seq_len)
        # 也要攤平成 (batch * seq_len)
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y_batch.reshape(-1)
        )

        # 反向傳播：計算每個參數的梯度
        loss.backward()

        # 用 optimizer 根據梯度更新參數
        optimizer.step()

        # 把這個 batch 的 loss 加到 total_loss
        total_loss += loss.item()

    # 計算這個 epoch 的平均 loss
    avg_loss = total_loss / len(dataloader)

    # 每 10 個 epoch 印一次，或第一個 epoch 就印
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")



Epoch [1/50], Loss: 2.2449
Epoch [10/50], Loss: 0.1050
Epoch [20/50], Loss: 0.0896
Epoch [30/50], Loss: 0.0883
Epoch [40/50], Loss: 0.0865
Epoch [50/50], Loss: 0.0839


## 5. 文字生成函式


In [ ]:
# 定義一個文字生成函式
# model: 訓練好的模型
# start_text: 生成的起始字串
# length: 額外要生成多少個字元
# temperature: 控制隨機性，越低越保守，越高越隨機
def generate_text(model, start_text, length=100, temperature=1.0):
    # 切換模型到推論模式
    model.eval()

    # 把起始字串 start_text 轉成 index list
    input_ids = [char_to_idx[ch] for ch in start_text]

    # 把 index list 轉成 tensor，外面再包一層 []
    # 是因為模型預期輸入 shape 為 (batch, seq_len)
    # 這裡 batch=1
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # 先把輸出字串初始化為 start_text
    generated = start_text

    # 初始化 hidden state
    # 先設成 None，讓 RNN 自己建立初始 hidden state
    h = None

    # 不需要計算梯度，節省記憶體與運算
    with torch.no_grad():
        # 先把整個 start_text 丟進模型
        # 目的：更新 hidden state，讓模型「記住」起始內容
        outputs, h = model(input_tensor, h)

    # 取 start_text 最後一個字元，作為下一步生成的輸入
    # input_tensor[0, -1] 代表 batch 中第 0 筆資料的最後一個字元
    # view(1, 1) 把 shape 調整成 (batch=1, seq_len=1)
    last_char = input_tensor[0, -1].view(1, 1)

    # 連續生成 length 個字元
    for _ in range(length):
        # 推論時不需要梯度
        with torch.no_grad():
            # 把上一個字元餵進模型
            # 並沿用前一步的 hidden state
            outputs, h = model(last_char, h)

            # 取出最後一個時間步的輸出 logits
            # outputs shape = (1, 1, vocab_size)
            # outputs[:, -1, :] shape = (1, vocab_size)
            #
            # 再除以 temperature，調整分布尖銳程度
            logits = outputs[:, -1, :] / temperature

            # 把 logits 轉成機率分布
            probs = torch.softmax(logits, dim=-1)

            # 根據機率分布隨機抽樣一個字元 index
            # 不是直接 argmax，所以生成會保留一些隨機性
            next_id = torch.multinomial(probs, num_samples=1).item()

            # 把 index 轉回對應字元
            next_char = idx_to_char[next_id]

            # 把新字元接到結果字串後面
            generated += next_char

            # 把剛生成的新字元轉成下一輪輸入
            # shape 同樣要是 (1, 1)
            last_char = torch.tensor([[next_id]], dtype=torch.long).to(device)

    # 回傳完整生成結果
    return generated

## 6. 測試生成


In [ ]:
# 印出提示文字
print("\nGenerated text:")

# 呼叫 generate_text 進行文字生成
# start_text="hello " 表示從 hello 加空白開始續寫
# length=120 表示再生成 120 個字元
# temperature=0.8 代表稍微保守一點，但仍保留隨機性
print(generate_text(model, start_text="hello ", length=120, temperature=0.8))


Generated text:
hello example.
rnn can learn next character prediction.
hello world.
this is a simple example.
rnn can learn next character pr


# LSTM Long Short-Term Memory

In [ ]:
# 匯入 PyTorch 主套件
# torch 是整個深度學習運算的核心，包含 tensor、GPU 運算等功能
import torch

# 匯入 PyTorch 的神經網路模組，簡寫成 nn
# 這裡面有 LSTM、Linear、Embedding、Loss Function 等元件
import torch.nn as nn

# 從 torch.utils.data 匯入 Dataset 與 DataLoader
# Dataset 用來定義資料集格式
# DataLoader 用來批次讀取資料
from torch.utils.data import Dataset, DataLoader


# =========================================================
# 1. 準備訓練資料
# =========================================================

# 準備一小段訓練文字
# 這裡故意重複幾次，讓模型比較容易學到規律
# lower() 會把所有字母轉成小寫，讓字元種類變少，學習更簡單
text = """
hello world.
this is a simple example.
lstm can learn next character prediction.
hello world.
this is a simple example.
lstm can learn next character prediction.
hello world.
this is a simple example.
lstm can learn next character prediction.
""".lower()

# 取出 text 中所有不重複的字元，並排序
# 例如可能包含：空白、換行、h、e、l、o、w、r、d、.、a、b...
chars = sorted(list(set(text)))

# vocab_size 表示總共有多少種不同字元
vocab_size = len(chars)

# 建立字典：字元 -> 整數索引
# 例如：
# 'a' -> 0
# 'b' -> 1
# 'c' -> 2
char_to_idx = {ch: i for i, ch in enumerate(chars)}

# 建立反向字典：整數索引 -> 字元
# 之後模型預測出數字，就可以轉回字元
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# 印出字元表，方便確認有哪些字元
print("Vocabulary:", chars)

# 印出字元總數
print("Vocab size:", vocab_size)

# 把整段文字轉成數字序列
# 例如 "hello" 可能會變成 [7, 4, 9, 9, 11]
encoded_text = [char_to_idx[ch] for ch in text]


# =========================================================
# 2. 建立自訂 Dataset
# =========================================================

# 建立一個字元資料集類別，繼承 PyTorch 的 Dataset
class CharDataset(Dataset):
    # 初始化函式
    # data: 整段已編碼的文字（整數序列）
    # seq_len: 每筆輸入序列的長度
    def __init__(self, data, seq_len):
        # 把資料存到物件裡
        self.data = data

        # 把序列長度存到物件裡
        self.seq_len = seq_len

    # 回傳資料集總共有幾筆樣本
    def __len__(self):
        # 假設資料長度是 N，每筆樣本會取 seq_len 個輸入
        # 並且還需要對應的下一個字元，所以能取的起始位置數量為：
        return len(self.data) - self.seq_len

    # 定義如何取得第 idx 筆資料
    def __getitem__(self, idx):
        # x 是輸入序列
        # 從 idx 開始取 seq_len 個字元
        x = self.data[idx: idx + self.seq_len]

        # y 是目標序列
        # 往右平移一個位置，表示要預測下一個字元
        y = self.data[idx + 1: idx + self.seq_len + 1]

        # 把 x 轉成 LongTensor
        # 為什麼是 long？
        # 因為 embedding 的輸入必須是整數索引
        x_tensor = torch.tensor(x, dtype=torch.long)

        # 把 y 也轉成 LongTensor
        # CrossEntropyLoss 的標籤也需要是 long 型別
        y_tensor = torch.tensor(y, dtype=torch.long)

        # 回傳一筆資料：(輸入序列, 目標序列)
        return x_tensor, y_tensor


# 設定每筆輸入序列長度
# 例如 seq_len = 20，表示模型一次看 20 個字元
seq_len = 20

# 建立資料集物件
dataset = CharDataset(encoded_text, seq_len)

# 建立 DataLoader
# batch_size=16 表示每次取 16 筆資料
# shuffle=True 表示每個 epoch 打亂資料順序
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


# =========================================================
# 3. 定義 LSTM 模型
# =========================================================

# 建立字元級 LSTM 模型，繼承 nn.Module
class CharLSTM(nn.Module):
    # 初始化模型
    # vocab_size: 字元種類數
    # embed_size: 每個字元 embedding 向量維度
    # hidden_size: LSTM 隱藏狀態維度
    # num_layers: LSTM 疊幾層
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        # 呼叫父類別初始化
        super().__init__()

        # 建立 Embedding 層
        # 功能：把字元索引轉成稠密向量
        # 輸入 shape: (batch, seq_len)
        # 輸出 shape: (batch, seq_len, embed_size)
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # 建立 LSTM 層
        # input_size=embed_size，因為輸入是 embedding 向量
        # hidden_size=hidden_size，表示隱藏狀態維度
        # num_layers=1，表示只有一層 LSTM
        # batch_first=True，表示輸入格式為 (batch, seq_len, feature)
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        # 建立全連接層
        # 把每個時間步的 hidden state 轉成 vocab_size 維的分數
        # 也就是每個字元類別各有一個分數
        self.fc = nn.Linear(hidden_size, vocab_size)

    # 定義前向傳播
    # x: 輸入 shape = (batch, seq_len)
    # hidden: LSTM 的隱藏狀態，包含 (h0, c0)
    def forward(self, x, hidden=None):
        # 先把字元索引轉成 embedding 向量
        # shape: (batch, seq_len) -> (batch, seq_len, embed_size)
        x = self.embedding(x)

        # 把 embedding 後的序列送進 LSTM
        # out: 每個時間步的輸出
        # hidden: 最後的 hidden state 與 cell state
        # out shape = (batch, seq_len, hidden_size)
        # hidden = (h_n, c_n)
        out, hidden = self.lstm(x, hidden)

        # 把每個時間步的輸出丟進全連接層
        # shape: (batch, seq_len, hidden_size) -> (batch, seq_len, vocab_size)
        out = self.fc(out)

        # 回傳每個時間步的分類分數，以及最後 hidden 狀態
        return out, hidden


# 判斷目前是否可使用 GPU
# 如果可用就使用 CUDA，否則退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 建立模型實例
# embed_size=16 表示每個字元轉成 16 維向量
# hidden_size=64 表示 LSTM 隱藏層維度為 64
# num_layers=1 表示使用 1 層 LSTM
model = CharLSTM(
    vocab_size=vocab_size,
    embed_size=16,
    hidden_size=64,
    num_layers=1
).to(device)

# 定義損失函數
# CrossEntropyLoss 適合多分類問題
# 這裡每個時間步都要從 vocab_size 個字元裡選出正確答案
criterion = nn.CrossEntropyLoss()

# 定義優化器 Adam
# lr=0.01 是學習率
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# =========================================================
# 4. 訓練模型
# =========================================================

# 設定總訓練回合數
epochs = 50

# 開始訓練
for epoch in range(epochs):
    # 切換到訓練模式
    model.train()

    # 累積這個 epoch 的總 loss
    total_loss = 0

    # 從 dataloader 一批一批讀出資料
    for x_batch, y_batch in dataloader:
        # 把輸入資料移到 device（CPU 或 GPU）
        x_batch = x_batch.to(device)

        # 把目標資料移到 device（CPU 或 GPU）
        y_batch = y_batch.to(device)

        # 清空上一輪累積的梯度
        optimizer.zero_grad()

        # 前向傳播
        # outputs shape = (batch, seq_len, vocab_size)
        outputs, hidden = model(x_batch)

        # 計算 loss
        # CrossEntropyLoss 需要：
        # 預測 shape = (N, C)
        # 標籤 shape = (N)
        #
        # 所以這裡要把 outputs 攤平成：
        # (batch * seq_len, vocab_size)
        #
        # 把 y_batch 攤平成：
        # (batch * seq_len)
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y_batch.reshape(-1)
        )

        # 反向傳播，計算梯度
        loss.backward()

        # 根據梯度更新模型參數
        optimizer.step()

        # 把這一批的 loss 累加起來
        total_loss += loss.item()

    # 計算平均 loss
    avg_loss = total_loss / len(dataloader)

    # 每 10 個 epoch 印一次結果
    # epoch == 0 也印，方便看訓練剛開始的狀態
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {avg_loss:.4f}")


# =========================================================
# 5. 文字生成函式
# =========================================================

# 定義生成文字的函式
# model: 訓練好的模型
# start_text: 起始字串
# length: 額外生成多少個字元
# temperature: 控制隨機程度
def generate_text(model, start_text, length=100, temperature=1.0):
    # 切換成推論模式
    model.eval()

    # 把起始文字轉成 index 序列
    input_ids = [char_to_idx[ch] for ch in start_text]

    # 把 index 序列轉成 tensor
    # 外面加一層 [] 是因為模型輸入需要 batch 維度
    # shape 會變成 (1, seq_len)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # 先把已知起始字串放進結果
    generated = start_text

    # 一開始 hidden state 設成 None
    # 讓 LSTM 自動初始化
    hidden = None

    # 不需要計算梯度，節省記憶體與加快推論
    with torch.no_grad():
        # 把整段 start_text 先送進模型
        # 這樣 LSTM 就能根據起始內容建立上下文記憶
        outputs, hidden = model(input_tensor, hidden)

    # 取起始字串最後一個字元，作為下一步輸入
    # input_tensor[0, -1] 表示第 0 筆資料的最後一個字元
    # view(1, 1) 把它調整成 shape = (1, 1)
    last_char = input_tensor[0, -1].view(1, 1)

    # 連續生成指定長度的字元
    for _ in range(length):
        # 推論不需要梯度
        with torch.no_grad():
            # 把上一個字元與 hidden state 一起送進模型
            outputs, hidden = model(last_char, hidden)

            # 取最後一個時間步的輸出 logits
            # outputs shape = (1, 1, vocab_size)
            # outputs[:, -1, :] shape = (1, vocab_size)
            logits = outputs[:, -1, :] / temperature

            # 用 softmax 把 logits 轉成機率分布
            probs = torch.softmax(logits, dim=-1)

            # 根據機率分布隨機抽樣出下一個字元的 index
            # num_samples=1 表示抽一個
            next_id = torch.multinomial(probs, num_samples=1).item()

            # 把 index 轉回字元
            next_char = idx_to_char[next_id]

            # 把新字元接到生成字串後面
            generated += next_char

            # 把剛生成的字元當成下一輪輸入
            # shape 必須是 (1, 1)
            last_char = torch.tensor([[next_id]], dtype=torch.long).to(device)

    # 回傳生成結果
    return generated


# =========================================================
# 6. 測試文字生成
# =========================================================

# 印出提示
print("\nGenerated text:")

# 呼叫文字生成函式
# start_text="hello " 表示從 hello + 空白開始
# length=120 表示再生成 120 個字元
# temperature=0.8 表示稍微保守一些，但仍保有隨機性
result = generate_text(model, start_text="hello ", length=120, temperature=0.8)

# 印出生成結果
print(result)

Vocabulary: ['\n', ' ', '.', 'a', 'c', 'd', 'e', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'w', 'x']
Vocab size: 19
Epoch [1/50], Loss: 2.4414
Epoch [10/50], Loss: 0.0926
Epoch [20/50], Loss: 0.0877
Epoch [30/50], Loss: 0.0854
Epoch [40/50], Loss: 0.0846
Epoch [50/50], Loss: 0.0841

Generated text:
hello world.
this is a simple example.
lstm can learn next character prediction.
hello world.
this is a simple example.
lstm c


# GRU Gated Recurrent Unit

In [ ]:
# 匯入 PyTorch 主套件
# torch 提供 tensor、GPU 運算、自動微分等核心功能
import torch

# 匯入神經網路模組，並簡寫成 nn
# 這裡面有 Embedding、GRU、Linear、Loss Function 等元件
import torch.nn as nn

# 匯入 Dataset 與 DataLoader
# Dataset 用來定義資料集格式
# DataLoader 用來批次讀取資料
from torch.utils.data import Dataset, DataLoader


# =========================================================
# 1. 準備訓練文字資料
# =========================================================

# 準備一小段訓練文字
# lower() 會把英文字母全部轉成小寫，減少字元種類
text = """
hello world.
this is a simple example.
gru can learn next character prediction.
hello world.
this is a simple example.
gru can learn next character prediction.
hello world.
this is a simple example.
gru can learn next character prediction.
""".lower()

# 取出所有不重複字元，並排序
# 例如可能包含：空白、換行、h、e、l、o、g、r、u、.
chars = sorted(list(set(text)))

# vocab_size 表示總共有多少種不同字元
vocab_size = len(chars)

# 建立字典：字元 -> 整數索引
# 例如 'a' -> 0, 'b' -> 1 ...
char_to_idx = {ch: i for i, ch in enumerate(chars)}

# 建立反向字典：整數索引 -> 字元
# 之後模型預測出 index，可以靠它轉回字元
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# 印出字元表
print("Vocabulary:", chars)

# 印出字元總數
print("Vocab size:", vocab_size)

# 把整段文字轉成整數序列
# 例如 "hello" 可能被轉成 [7, 4, 9, 9, 11]
encoded_text = [char_to_idx[ch] for ch in text]


# =========================================================
# 2. 建立自訂資料集 Dataset
# =========================================================

# 建立字元資料集類別，繼承 Dataset
class CharDataset(Dataset):
    # 初始化函式
    # data: 已經編碼好的整數序列
    # seq_len: 每筆樣本的輸入長度
    def __init__(self, data, seq_len):
        # 把資料存起來
        self.data = data

        # 把序列長度存起來
        self.seq_len = seq_len

    # 定義資料集總長度
    def __len__(self):
        # 如果總長度是 N，每筆資料要用到 seq_len 個輸入位置
        # 所以最多可取 len(data) - seq_len 筆
        return len(self.data) - self.seq_len

    # 定義如何取出第 idx 筆資料
    def __getitem__(self, idx):
        # x 是輸入序列
        # 從 idx 開始取 seq_len 個字元
        x = self.data[idx: idx + self.seq_len]

        # y 是目標序列
        # 往右平移 1 個位置，表示要預測下一個字元
        y = self.data[idx + 1: idx + self.seq_len + 1]

        # 把 x 轉成 LongTensor
        # Embedding 需要整數型別索引
        x_tensor = torch.tensor(x, dtype=torch.long)

        # 把 y 轉成 LongTensor
        # CrossEntropyLoss 的標籤也需要 long
        y_tensor = torch.tensor(y, dtype=torch.long)

        # 回傳一筆資料：(輸入, 目標)
        return x_tensor, y_tensor


# 設定輸入序列長度
# 例如一次看 20 個字元
seq_len = 20

# 建立資料集物件
dataset = CharDataset(encoded_text, seq_len)

# 建立 DataLoader
# batch_size=16 表示每批 16 筆資料
# shuffle=True 表示每個 epoch 都打亂資料
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


# =========================================================
# 3. 定義 GRU 模型
# =========================================================

# 建立字元級 GRU 模型
class CharGRU(nn.Module):
    # 初始化函式
    # vocab_size: 字元總數
    # embed_size: embedding 維度
    # hidden_size: GRU 隱藏層維度
    # num_layers: GRU 層數
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        # 呼叫父類別初始化
        super().__init__()

        # 建立 Embedding 層
        # 把字元索引轉成稠密向量
        # 輸入 shape: (batch, seq_len)
        # 輸出 shape: (batch, seq_len, embed_size)
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # 建立 GRU 層
        # input_size=embed_size，因為輸入是 embedding 向量
        # hidden_size=hidden_size，表示 GRU 隱藏狀態大小
        # num_layers=1，表示用一層 GRU
        # batch_first=True，表示輸入格式為 (batch, seq_len, feature)
        self.gru = nn.GRU(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        # 建立全連接層
        # 把每個時間步的 hidden state 轉成 vocab_size 維分數
        # 代表每個字元類別各自的 logits
        self.fc = nn.Linear(hidden_size, vocab_size)

    # 定義前向傳播
    # x: 輸入 shape = (batch, seq_len)
    # h: 初始 hidden state，預設 None
    def forward(self, x, h=None):
        # 先經過 Embedding
        # shape: (batch, seq_len) -> (batch, seq_len, embed_size)
        x = self.embedding(x)

        # 把 embedding 序列送進 GRU
        # out: 每個時間步的輸出
        # h: 最後的 hidden state
        # out shape = (batch, seq_len, hidden_size)
        # h shape = (num_layers, batch, hidden_size)
        out, h = self.gru(x, h)

        # 經過全連接層，把 hidden state 轉成字元分類分數
        # shape: (batch, seq_len, hidden_size) -> (batch, seq_len, vocab_size)
        out = self.fc(out)

        # 回傳每個時間步的輸出分數，以及最後的 hidden state
        return out, h


# 判斷是否可使用 GPU
# 若有 CUDA 則用 GPU，否則用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 建立模型
# embed_size=16: 每個字元轉成 16 維向量
# hidden_size=64: GRU 隱藏狀態維度 64
# num_layers=1: 一層 GRU
model = CharGRU(
    vocab_size=vocab_size,
    embed_size=16,
    hidden_size=64,
    num_layers=1
).to(device)

# 定義損失函數
# 這裡是多分類問題，所以使用 CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# 定義優化器 Adam
# lr=0.01 是學習率
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# =========================================================
# 4. 訓練模型
# =========================================================

# 設定總訓練回合數
epochs = 50

# 開始訓練
for epoch in range(epochs):
    # 切換模型到訓練模式
    model.train()

    # 累積這個 epoch 的總 loss
    total_loss = 0

    # 逐批讀取資料
    for x_batch, y_batch in dataloader:
        # 把輸入資料移到 CPU 或 GPU
        x_batch = x_batch.to(device)

        # 把標籤資料移到 CPU 或 GPU
        y_batch = y_batch.to(device)

        # 清空上一輪累積的梯度
        optimizer.zero_grad()

        # 前向傳播
        # outputs shape = (batch, seq_len, vocab_size)
        outputs, h = model(x_batch)

        # 計算 loss
        # CrossEntropyLoss 需要：
        # 預測 shape = (N, C)
        # 標籤 shape = (N)
        #
        # 所以把 outputs 攤平成 (batch * seq_len, vocab_size)
        # 把 y_batch 攤平成 (batch * seq_len)
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y_batch.reshape(-1)
        )

        # 反向傳播，計算梯度
        loss.backward()

        # 根據梯度更新模型參數
        optimizer.step()

        # 把這個 batch 的 loss 累加
        total_loss += loss.item()

    # 計算這個 epoch 的平均 loss
    avg_loss = total_loss / len(dataloader)

    # 每 10 個 epoch 印一次
    # 或第 1 個 epoch 先印一次
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {avg_loss:.4f}")


# =========================================================
# 5. 文字生成函式
# =========================================================

# 定義文字生成函式
# model: 訓練好的模型
# start_text: 起始字串
# length: 額外生成字元數
# temperature: 控制隨機程度
def generate_text(model, start_text, length=100, temperature=1.0):
    # 切換成推論模式
    model.eval()

    # 把起始字串轉成 index 序列
    input_ids = [char_to_idx[ch] for ch in start_text]

    # 轉成 tensor，並加上 batch 維度
    # shape: (1, seq_len)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # 先把起始字串放進結果
    generated = start_text

    # 初始化 hidden state
    # 設成 None，讓 GRU 自動建立初始 hidden
    h = None

    # 推論不需要梯度
    with torch.no_grad():
        # 先把整段起始文字送進模型
        # 目的是建立上下文 hidden state
        outputs, h = model(input_tensor, h)

    # 取最後一個字元作為下一步輸入
    # shape 調整成 (1, 1)
    last_char = input_tensor[0, -1].view(1, 1)

    # 連續生成指定數量的字元
    for _ in range(length):
        # 推論不需要梯度
        with torch.no_grad():
            # 把上一個字元與目前 hidden state 送進模型
            outputs, h = model(last_char, h)

            # 取最後一個時間步的輸出 logits
            # outputs shape = (1, 1, vocab_size)
            # outputs[:, -1, :] shape = (1, vocab_size)
            logits = outputs[:, -1, :] / temperature

            # 把 logits 轉成機率分布
            probs = torch.softmax(logits, dim=-1)

            # 根據機率分布抽樣一個下一字元
            next_id = torch.multinomial(probs, num_samples=1).item()

            # 把 index 轉回字元
            next_char = idx_to_char[next_id]

            # 接到結果字串後面
            generated += next_char

            # 把新生成的字元當作下一輪輸入
            last_char = torch.tensor([[next_id]], dtype=torch.long).to(device)

    # 回傳生成結果
    return generated


# =========================================================
# 6. 測試生成結果
# =========================================================

# 印出提示文字
print("\nGenerated text:")

# 以 "hello " 當起始字串，生成後續文字
result = generate_text(model, start_text="hello ", length=120, temperature=0.8)

# 印出生成結果
print(result)

Vocabulary: ['\n', ' ', '.', 'a', 'c', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'x']
Vocab size: 21
Epoch [1/50], Loss: 2.2814
Epoch [10/50], Loss: 0.0859
Epoch [20/50], Loss: 0.0839
Epoch [30/50], Loss: 0.0831
Epoch [40/50], Loss: 0.0828
Epoch [50/50], Loss: 0.0820

Generated text:
hello world.
this is a simple example.
gru can learn next character prediction.
hello world.
this is a simple example.
gru can
